# Demo: Working with Time-Series Data in pandas

We've got three years of daily bike-share ride counts. Before we can do anything useful with this data -- forecast it, model it, decompose it -- we need to get the time index right. pandas has great support for datetime types, but it won't do the work for you. You have to tell it which column is a date and set it as your index. That's what this demo is about.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Loading and Parsing Dates

Let's load the CSV and see what we get by default, then fix it.

In [ ]:
# Without parse_dates -- pandas treats the date column as a plain string
df_raw = pd.read_csv("../data/bikeshare_rides.csv")
print(f"date column type: {df_raw['date'].dtype}")
print(f"Index type: {type(df_raw.index)}")
df_raw.head()

See the problem? The date column is an `object` -- just a string. And the index is a boring `RangeIndex`. pandas doesn't know this is time-series data. Let's fix both at once.

In [ ]:
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
df = df.set_index("date")
print(f"Index type: {type(df.index)}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Rows: {len(df)}")
df.head()

Now we have a `DatetimeIndex`. This unlocks pandas' time-series capabilities -- resampling, time-based slicing, frequency detection. Without it, none of that works.

## Checking for Gaps

Three years of daily data should be about 1,096 rows. We have a different number. Let's find out where the gaps are.

In [ ]:
full_range = pd.date_range(df.index.min(), df.index.max(), freq="D")
missing = full_range.difference(df.index)
print(f"Expected: {len(full_range)} days")
print(f"Actual: {len(df)} rows")
print(f"Missing dates: {len(missing)}")
for d in missing:
    print(f"  {d.date()}")

We also have more rows than expected dates, which means there are duplicates somewhere. You'd only catch this by checking the index -- summary statistics won't help.

## Resampling

One of the things a DatetimeIndex gives you is `.resample()`. This lets you change the frequency of your data.

In [ ]:
weekly = df["rides"].resample("W").sum()
monthly = df["rides"].resample("MS").sum()

print(f"Daily:   {len(df)} rows")
print(f"Weekly:  {len(weekly)} rows")
print(f"Monthly: {len(monthly)} rows")

## Plotting at Different Frequencies

When you resample, you trade detail for clarity. Daily data is noisy. Monthly data is smooth. The right frequency depends on the question you're answering.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df.index, df["rides"].values, linewidth=0.5, color="tab:blue")
axes[0].set_ylabel("Rides")
axes[0].set_title("Daily")
axes[1].plot(weekly.index, weekly.values, linewidth=0.8, color="tab:orange")
axes[1].set_ylabel("Rides")
axes[1].set_title("Weekly")
axes[2].plot(monthly.index, monthly.values, linewidth=1.2, color="black")
axes[2].set_ylabel("Rides")
axes[2].set_title("Monthly")
plt.tight_layout()
plt.show()

The monthly view makes the seasonal pattern obvious -- rides peak in summer and drop in winter. You can sort of see it in the daily data, but it's buried in noise. Getting the time index right and knowing how to resample is always step one.